# Testing Llama / Qwen / Gemma / Mistral on `merged_all_labeled.csv`
Two complementary tests, one model loaded at a time (4-bit, free Colab **T4 GPU**):

**Test A - the model answers.** Questions go to each model and its answers are scored against gold: wrong answers = hallucination; answering a SQuAD-unanswerable question = hallucination; on AmbigQA multi-answer questions, does it notice the ambiguity or cover several interpretations?

**Test B - the model judges.** Each model labels rows as hallucinated / ambiguous / hard-to-understand, scored against the CSV's `auto_halluc_label`, `auto_ambiguous`, `auto_grammar_label`.

**Before you run:** (1) Runtime > Change runtime type > T4 GPU. (2) Gated models: accept the license on each model's Hugging Face page, create a read token, and add it in Colab's Secrets tab as `HF_TOKEN`. (3) Start with the default small sample, then set `N_PER_GROUP = None` for the full run.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes scikit-learn datasets
import os, re, json, gc, string, random
import numpy as np, pandas as pd, torch
from google.colab import files
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # Colab "Secrets" tab -> add HF_TOKEN
except Exception as e:
    print("No HF_TOKEN secret found; gated models (Llama, Gemma, Mistral) will fail to download.", e)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch Runtime > Change runtime type > T4 GPU")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
No HF_TOKEN secret found; gated models (Llama, Gemma, Mistral) will fail to download. Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.
GPU: Tesla T4


## 1. Config and data

In [2]:
# ---------------- CONFIG ----------------
MODELS = {   # swap in newer checkpoints if you prefer; all are loaded in 4-bit to fit a free T4
    "llama":   "meta-llama/Llama-3.1-8B-Instruct",
    "qwen":    "Qwen/Qwen2.5-7B-Instruct",
    "gemma":   "google/gemma-2-9b-it",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",
}
N_PER_GROUP = 40      # rows per group (smoke test). Set to None to run ALL rows (slow: hours on a T4)
SEED = 0

uploaded = files.upload()          # upload merged_all_labeled.csv
df = pd.read_csv(next(iter(uploaded)))
print(df.shape)

Saving merged_all_labeled.csv to merged_all_labeled.csv
(1500, 32)


## 2. (Optional) Re-attach HaluBench passages
Without the passage, hallucination judging on HaluBench rows is closed-book guessing, so treat those scores with care.

In [3]:
# OPTIONAL: the CSV has passages only for squad_v2. HaluBench rows have no context, so a model cannot judge
# "is this answer supported by the passage?". Try to re-attach the original passages from the public dataset.
try:
    from datasets import load_dataset
    hb = load_dataset("PatronusAI/HaluBench", split="test").to_pandas()
    ctx = dict(zip(hb["id"].astype(str), hb["context"]))
    m = df.source.str.startswith("halubench") & df.context.isna()
    new = df.loc[m, "id"].str.replace("^halubench_", "", regex=True).map(ctx)
    df.loc[m, "context"] = new
    print("HaluBench contexts re-attached:", int(new.notna().sum()), "of", int(m.sum()))
except Exception as e:
    print("Could not attach HaluBench contexts (continuing closed-book for those rows):", e)

README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.51MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/14900 [00:00<?, ? examples/s]

Could not attach HaluBench contexts (continuing closed-book for those rows): 'context'


## 3. Helpers

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def run_model(model_id, jobs, bs=8):
    """jobs: list of (prompt, max_new_tokens). Loads the model once, returns list of outputs, frees the GPU."""
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
              bnb_4bit_compute_dtype=torch.float16), device_map="auto")
    if "gemma-2" in model_id: kw["attn_implementation"] = "eager"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    outs = [None]*len(jobs)
    order = sorted(range(len(jobs)), key=lambda i: (jobs[i][1], len(jobs[i][0])))   # batch similar jobs together
    for s in range(0, len(order), bs):
        idx = order[s:s+bs]; mx = max(jobs[i][1] for i in idx)
        texts = [tok.apply_chat_template([{"role":"user","content":jobs[i][0]}], tokenize=False, add_generation_prompt=True) for i in idx]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=mx, do_sample=False, pad_token_id=tok.pad_token_id)
        dec = tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for i, d in zip(idx, dec): outs[i] = d.strip()
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return outs

In [5]:
def norm(s):
    s = str(s).lower(); s = "".join(ch for ch in s if ch not in string.punctuation)
    return re.sub(r"\b(a|an|the)\b", " ", s).split()
def f1(p, g):
    from collections import Counter
    c = sum((Counter(p) & Counter(g)).values())
    if not c: return 0.0
    pr, rc = c/len(p), c/len(g); return 2*pr*rc/(pr+rc)
def matches(pred, golds):
    p = norm(pred)
    for g in golds:
        gn = norm(g)
        if gn and (" ".join(gn) in " ".join(p) or f1(p, gn) >= 0.5): return True
    return False

def score_A(r, pred):
    low = pred.lower()
    abstain = "unanswerable" in low[:40] or low.strip(". ").startswith("unknown")
    flagged = low.lstrip().startswith("ambiguous")
    out = dict(abstained=abstain, flagged_ambiguous=flagged, correct=np.nan, coverage=np.nan)
    if r.source == "squad_v2" and r.ref_squad_unanswerable == "Yes":
        out["hallucinated"] = not abstain                       # answered an unanswerable question
    else:
        golds = str(r.answer).split(" | ")
        out["correct"] = matches(pred, golds)
        if r.ref_ambigqa_type == "multipleQAs":
            qas = json.loads(r.ref_ambigqa_disambiguated)
            out["coverage"] = float(np.mean([matches(pred, qa["answer"]) for qa in qas]))
            out["correct"] = out["coverage"] > 0
        out["hallucinated"] = (not abstain) and (not out["correct"])   # confident but wrong
    return out

def yes_no(o):
    m = re.match(r"\W*(yes|no)\b", o, re.I); return m.group(1).capitalize() if m else None

## 4. Build the test sets

In [6]:
def sample(d, by, n):
    if n is None: return d
    return pd.concat([g.sample(min(len(g), n), random_state=SEED) for _, g in d.groupby(by, dropna=False)])

# ---- TEST A: let each model ANSWER (hallucination + ambiguity handling). Clean gold only for AmbigQA + SQuAD2 ----
A = df[df.source.isin(["ambigqa", "squad_v2"])].copy()
A["grp"] = A.source + "|" + A.ref_ambigqa_type.fillna(A.ref_squad_unanswerable.map({"Yes":"unanswerable","No":"answerable"}))
A = sample(A, "grp", N_PER_GROUP).reset_index(drop=True)

def prompt_A(r):
    if isinstance(r.context, str):
        return ("Answer the question using ONLY the context. If the context does not contain the answer, reply exactly: UNANSWERABLE\n\n"
                f"Context: {r.context}\n\nQuestion: {r.question}\nAnswer:"), 48
    return ("Answer the question in a few words. If the question is ambiguous and has several valid interpretations, reply with "
            "'AMBIGUOUS:' followed by each interpretation and its answer. If you do not know, reply exactly: UNKNOWN\n\n"
            f"Question: {r.question}\nAnswer:"), 120

# ---- TEST B: models as JUDGES of the three properties, scored against the CSV's auto_* labels ----
def sel(col, yes):  # rows that have a label for this task
    d = df[df[col].notna()].copy(); d["gold"] = d[col]; return sample(d, "source", N_PER_GROUP).reset_index(drop=True)

TASKS = {
 "hallucination": dict(data=sel("auto_halluc_label", "Yes"), positive="Yes",
    prompt=lambda r: ("You are fact-checking an answer.\n" +
        (f"Context:\n{r.context}\n\n" if isinstance(r.context, str) else "(No context is available; use your own knowledge.)\n\n") +
        f"Question: {r.question}\nProposed answer: {r.answer}\n\n"
        "Is the proposed answer a hallucination (unsupported by the context, or factually wrong)? Reply with exactly one word: Yes or No.")),
 "ambiguity": dict(data=sel("auto_ambiguous", "Yes"), positive="Yes",
    prompt=lambda r: (f"Question: {r.question}\n\nIs this question ambiguous, i.e. can it reasonably be interpreted in more than one way "
        "leading to different answers? Reply with exactly one word: Yes or No.")),
 "understandability": dict(data=sel("auto_grammar_label", "No"), positive="No",   # 'No' = has a grammar/clarity PROBLEM
    prompt=lambda r: (f"Text: {r.answer}\n\nIs this text grammatically correct and easy to understand? Reply with exactly one word: Yes or No.")),
}
print(len(A), "rows in test A;", {k: len(v["data"]) for k, v in TASKS.items()}, "rows in test B tasks")

160 rows in test A; {'hallucination': 277, 'ambiguity': 68, 'understandability': 217} rows in test B tasks


## 5. Run all four models (~15-40 min per model on the default sample)

In [10]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
resA, resB = [], []
for name, mid in MODELS.items():
    print(f"\n===== {name}: {mid} =====")
    jobs = [prompt_A(r) for r in A.itertuples()]
    keys = [("A", i) for i in range(len(A))]
    for t, spec in TASKS.items():
        for i, r in enumerate(spec["data"].itertuples()):
            jobs.append((spec["prompt"](r), 4)); keys.append((t, i))
    try:
        outs = run_model(mid, jobs)
    except Exception as e:
        print("SKIPPED", name, "->", e); continue
    for (t, i), o in zip(keys, outs):
        if t == "A":
            resA.append(dict(model=name, id=A.id[i], source=A.source[i], grp=A.grp[i], pred=o, **score_A(A.iloc[i], o)))
        else:
            d = TASKS[t]["data"]
            resB.append(dict(model=name, task=t, id=d.id[i], source=d.source[i], gold=d.gold[i], pred=yes_no(o), raw=o))
    pd.DataFrame(resA).to_csv("/content/results_answers.csv", index=False)
    pd.DataFrame(resB).to_csv("/content/results_judge.csv", index=False)
resA, resB = pd.DataFrame(resA), pd.DataFrame(resB)


===== llama: meta-llama/Llama-3.1-8B-Instruct =====
SKIPPED llama -> You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-6af5c05f-6fa95327318df7a341c6d462;04ed7509-cbc2-45ac-b216-991785953904)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/0e9e39f249a16976918f6564b8830bc894c89659/config.json.
Your request to access model meta-llama/Llama-3.1-8B-Instruct is awaiting a review from the repo authors.

===== qwen: Qwen/Qwen2.5-7B-Instruct =====


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


===== gemma: google/gemma-2-9b-it =====


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

... (5 outputs left out)

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]


===== mistral: mistralai/Mistral-7B-Instruct-v0.3 =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## 6. Results

In [11]:
pd.set_option("display.width", 200)
# ---- TEST A summary: does the model hallucinate / handle ambiguity when ANSWERING? ----
def summ(g):
    s = {}
    for grp, x in g.groupby("grp"):
        if grp.endswith("unanswerable"):  s[f"SQuAD unanswerable: hallucination rate"] = x.hallucinated.mean()
        elif grp.startswith("squad"):     s["SQuAD answerable: accuracy"] = x.correct.astype(float).mean(); s["SQuAD answerable: wrong-answer rate"] = x.hallucinated.mean()
        elif grp.endswith("singleAnswer"):s["AmbigQA single: accuracy"] = x.correct.astype(float).mean(); s["AmbigQA single: wrong-answer rate"] = x.hallucinated.mean(); s["AmbigQA single: false 'ambiguous' flag rate"] = x.flagged_ambiguous.mean()
        else:                             s["AmbigQA multi: flagged ambiguous"] = x.flagged_ambiguous.mean(); s["AmbigQA multi: mean interpretation coverage"] = x.coverage.mean(); s["AmbigQA multi: gave >=1 valid answer"] = x.correct.astype(float).mean()
    return pd.Series(s)
print("TEST A - model answers"); display(resA.groupby("model").apply(summ).round(3).T)

# ---- TEST B summary: can the model DETECT the problem? (P/R/F1 are for the problem class) ----
rows = []
for (m, t), g in resB.groupby(["model", "task"]):
    pos = TASKS[t]["positive"]; ok = g[g.pred.notna()]
    p, r, f, _ = precision_recall_fscore_support(ok.gold == pos, ok.pred == pos, average="binary", zero_division=0)
    rows.append(dict(model=m, task=t, n=len(g), unparseable=1-len(ok)/len(g), accuracy=accuracy_score(ok.gold, ok.pred), precision=p, recall=r, f1=f))
print("TEST B - model as judge vs CSV labels"); display(pd.DataFrame(rows).round(3))
print("Accuracy by source:"); display(resB.assign(hit=resB.gold == resB.pred).pivot_table(index=["task", "source"], columns="model", values="hit").round(2))

TEST A - model answers


/tmp/ipykernel_444/2547499469.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("TEST A - model answers"); display(resA.groupby("model").apply(summ).round(3).T)


model,gemma,mistral,qwen
AmbigQA multi: flagged ambiguous,0.550,0.375,0.150
AmbigQA multi: mean interpretation coverage,0.182,0.213,0.058
AmbigQA multi: gave >=1 valid answer,0.400,0.525,0.125
AmbigQA single: accuracy,0.375,0.375,0.175
AmbigQA single: wrong-answer rate,0.500,0.625,0.150
AmbigQA single: false 'ambiguous' flag rate,0.400,0.300,0.075
SQuAD answerable: accuracy,0.800,0.675,0.800
SQuAD answerable: wrong-answer rate,0.100,0.100,0.100
SQuAD unanswerable: hallucination rate,0.350,0.225,0.150


TEST B - model as judge vs CSV labels


,model,task,n,unparseable,accuracy,precision,recall,f1
0,gemma,ambiguity,68,0.0,0.529,0.522,1.000,0.686
1,gemma,hallucination,277,0.0,0.527,0.388,0.758,0.513
2,gemma,understandability,217,0.0,0.885,0.182,0.364,0.242
3,mistral,ambiguity,68,0.0,0.529,0.522,1.000,0.686
4,mistral,hallucination,277,0.0,0.534,0.403,0.868,0.551
5,mistral,understandability,217,0.0,0.945,0.000,0.000,0.000
6,qwen,ambiguity,68,0.0,0.574,0.575,0.657,0.613
7,qwen,hallucination,277,0.0,0.480,0.137,0.110,0.122
8,qwen,understandability,217,0.0,0.829,0.139,0.455,0.213


Accuracy by source:


model                                     gemma  mistral  qwen
task              source                                      
ambiguity         ambigqa                  0.65     0.62  0.55
                  halubench_DROP           1.00     1.00  1.00
                  halubench_covidQA        1.00     1.00  1.00
                  halubench_halueval       0.53     0.60  0.80
                  halubench_pubmedQA       0.00     0.00  0.00
                  squad_v2                 0.00     0.00  0.30
hallucination     ambigqa                  0.28     0.15  0.60
                  halubench_DROP           0.57     0.57  0.40
                  halubench_FinanceBench   0.54     0.54  0.51
                  halubench_RAGTruth       0.65     0.65  0.81
                  halubench_covidQA        0.26     0.37  0.48
                  halubench_halueval       0.57     0.57  0.40
                  halubench_pubmedQA       0.50     0.62  0.56
                  squad_v2                 0.80     0.80  0.15
understandability ambigqa                  0.88     0.95  0.65
                  halubench_DROP           0.00     1.00  0.00
                  halubench_FinanceBench   0.88     1.00  1.00
                  halubench_RAGTruth       0.94     0.94  0.94
                  halubench_covidQA        0.95     0.84  0.95
                  halubench_halueval       0.90     0.95  0.82
                  halubench_pubmedQA       0.97     1.00  0.97
                  squad_v2                 0.90     0.92  0.88

In [12]:
for f in ["results_answers.csv", "results_judge.csv"]: files.download("/content/" + f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## How to read the results
- **Hallucination rate (Test A)** is the headline number: answers that are wrong or invented. On SQuAD-unanswerable it is the share of questions where the model answered instead of saying UNANSWERABLE.
- **AmbigQA multi - flagged / coverage:** high flag rate with low false-flag rate on singleAnswer questions means the model recognises ambiguity.
- **Test B F1 on the problem class** shows how well a model can *detect* each problem; compare with the always-"No" baseline when labels are imbalanced.
- **Understandability** uses `auto_grammar_label`, which in this CSV grades the *answer text* (sentence-type answers only), so Test B asks the model about the answer text.
- Labels are mostly dataset-derived (`ref_*` columns) rather than human-verified, and AmbigQA gold answers are closed-book Wikipedia facts, so small models will miss some answers they genuinely don't know - read "wrong" as "wrong or unknown".